# Analysis of experiment #11: CFCP on open neighborhoods for DIMAC instances

## Read data

In [1]:
# Number of vertices and edges in the hypergraphs
import pandas as pd
hypergraphs={
    "name": ['david','huck','jean','myciel3','myciel4','myciel5','queen5_5','queen6_6','queen7_7'],
    "n_H": [87,74,80,11,23,47,25,36,49],
    "m_H": [78,72,66,11,23,47,25,36,49]
}
df_hypergraphs = pd.DataFrame(hypergraphs)
print(df_hypergraphs)

       name  n_H  m_H
0     david   87   78
1      huck   74   72
2      jean   80   66
3   myciel3   11   11
4   myciel4   23   23
5   myciel5   47   47
6  queen5_5   25   25
7  queen6_6   36   36
8  queen7_7   49   49


In [2]:
# Read csv

import pandas as pd
import numpy as np

df = pd.read_csv("stats11.csv")

# Add column with the density of the graph
df["density"] = df.apply(lambda row: 2 * row.nedges / (row.nvertices * (row.nvertices - 1)), axis=1)

# Join the hypergraph data to the main dataframe
df = df.join(df_hypergraphs.set_index("name"), on="instance", how="left")

# Remove final _ in solver name
df['solver'] = df['solver'].str.rstrip('_')

# Choose solvers to keep
df = df[df['solver'].isin(['byp', 'gurobi'])]

# Change solver name
df['solver'] = df['solver'].replace('byp', 'b&p')

# Sort values by instance and solver
df = df.sort_values(by=["instance", "solver"]).reset_index(drop=True)

# Unify TIME_EXCEEDED values
df = df.replace({'TIME_EXCEEDED_PR': 'TIME_EXCEEDED', 'TIME_EXCEEDED_LP': 'TIME_EXCEEDED'}) 

# Remove the heuristic time from the total time
df.time = df.time - df.initialHeurTime

# Format timelimit
df.loc[df.state == 'TIME_EXCEEDED', 'time'] = 3600

# Format number of nodes for cplex
df.loc[(df.solver == 'cplex') & (df.nodes == 0), 'nodes'] = 1

print(list(df.columns))
df.head()

['instance', 'solver', 'run', 'nvertices', 'nedges', 'nP', 'nQ', 'nvars', 'ncons', 'state', 'terminationReason', 'time', 'nodes', 'nodesLeft', 'lb', 'ub', 'gap', 'initialHeurValue', 'initialHeurTime', 'initialSemigreedyIters', 'nNodesInt', 'nNodesFrac', 'nNodesGcp', 'nNodesTrivial', 'nNodesInfeas', 'nNodesInfeasPrepro', 'nNodesInfeasCheck', 'nNodesInfeasAux', 'gcpAvgTime', 'nsol', 'nsolHeur', 'nsolLR', 'nsolGCP', 'nsolTrivial', 'ninitSol', 'ninitDummy', 'ninit', 'rootNVertices', 'rootNEdges', 'rootNP', 'rootNQ', 'rootlb', 'rootub', 'rootHeurTime', 'rootFeasTime', 'rootCgTime', 'rootNCalls', 'rootNCallsPool', 'rootNCallsHeur', 'rootNCallsMwis1', 'rootNCallsMwis2', 'rootNCallsExact', 'rootNCols', 'rootNColsPool', 'rootNColsHeur', 'rootNColsMwis1', 'rootNColsMwis2', 'rootNColsExact', 'rootTime', 'rootTimePool', 'rootTimeHeur', 'rootTimeMwis1', 'rootTimeMwis2', 'rootTimeExact', 'otherNodesHeurTime', 'otherNodesFeasNCalls', 'otherNodesFeasTime', 'otherNodesNCalls', 'otherNodesNCallsPool', '

,instance,solver,run,nvertices,nedges,nP,nQ,nvars,ncons,state,...,otherNodesNColsExact,otherNodesTime,otherNodesTimePool,otherNodesTimeHeur,otherNodesTimeMwis1,otherNodesTimeMwis2,otherNodesTimeExact,density,n_H,m_H
0,david,b&p,0,803,195050,78,87,-1,-1,OPTIMAL,...,0.0,1.69819,0.000475,0.156838,0.005558,1.535320,0.00000,0.605740,87,78
1,david,gurobi,0,803,195050,78,87,2412,675000,OPTIMAL,...,0.0,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000,0.605740,87,78
2,huck,b&p,0,599,82770,72,74,-1,-1,OPTIMAL,...,0.0,1.00259,0.001037,0.368706,0.131341,0.501503,0.00000,0.462141,74,72
3,huck,gurobi,0,599,82770,72,74,3588,742690,OPTIMAL,...,0.0,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000,0.462141,74,72
4,jean,b&p,0,495,46729,66,80,-1,-1,OPTIMAL,...,0.0,1.12861,0.000867,0.268176,0.138498,0.270624,0.45044,0.382194,80,66


In [3]:
# pivot table by solver
df2 = df.pivot(index=["instance", "nvertices", "density", "nP", "nQ", "initialHeurValue"], 
                columns="solver", 
                values=["time", "nodes", "lb", "ub", "state"],
                ).reset_index()
print(list(df2.columns))
df2.head()

[('instance', ''), ('nvertices', ''), ('density', ''), ('nP', ''), ('nQ', ''), ('initialHeurValue', ''), ('time', 'b&p'), ('time', 'gurobi'), ('nodes', 'b&p'), ('nodes', 'gurobi'), ('lb', 'b&p'), ('lb', 'gurobi'), ('ub', 'b&p'), ('ub', 'gurobi'), ('state', 'b&p'), ('state', 'gurobi')]


instance nvertices   density  nP  nQ initialHeurValue     time  \
solver                                                            b&p   
0         david       803  0.605740  78  87              3.0   12.483   
1          huck       599  0.462141  72  74              6.0   16.705   
2          jean       495  0.382194  66  80              4.0  1738.66   
3       myciel3        40  0.391026  11  11              2.0  0.00419   
4       myciel4       142  0.402557  23  23              2.0   0.0838   

                nodes          lb          ub           state           
solver   gurobi   b&p gurobi  b&p gurobi  b&p gurobi      b&p   gurobi  
0        37.152     3      1  2.0    2.0  2.0    2.0  OPTIMAL  OPTIMAL  
1       622.098    11      1  2.0    2.0  2.0    2.0  OPTIMAL  OPTIMAL  
2       306.606  1365    645  3.0    3.0  3.0    3.0  OPTIMAL  OPTIMAL  
3       0.03078     1      1  2.0    2.0  2.0    2.0  OPTIMAL  OPTIMAL  
4        1.2281     1      1  2.0    2.0  2.0    2.0  OPTIMAL  OPTIMAL

In [4]:
df3 = df2[[('instance',''), ('nvertices',''), ('density',''), ('nP',''), ('nQ',''), ('initialHeurValue',''), 
           ('time','b&p'), ('time','gurobi'), ('nodes','b&p'), ('nodes','gurobi'), 
           ('lb','b&p'), ('lb','gurobi'), ('ub','b&p'), ('ub','gurobi')]]
colNames = pd.MultiIndex.from_tuples([('instance',''), ('|V|',''), ('density',''), ('n',''), ('m',''), ('hval',''), 
           ('time (s)','b&p'), ('time (s)','gurobi'), ('nodes','b&p'), ('nodes','gurobi'), 
           ('lb','b&p'), ('lb','gurobi'), ('ub','b&p'), ('ub','gurobi')])
df3.columns = colNames
df3[('density','')] = df3[('density','')].round(2)
df3[('hval','')] = df3[('hval','')].astype(int)
df3[('nodes','b&p')] = df3[('nodes','b&p')].astype(int)
df3[('nodes','gurobi')] = df3[('nodes','gurobi')].astype(int)
df3[('lb','b&p')] = df3[('lb','b&p')].round(2)
df3[('lb','gurobi')] = df3[('lb','gurobi')].round(2)
df3[('ub','b&p')] = df3[('ub','b&p')].astype(int)
df3[('ub','gurobi')] = df3[('ub','gurobi')].astype(int)

# Specific formatting
df3[('time (s)','b&p')] = df2.apply(lambda row: "tilim" if row[('state','b&p')] == 'TIME_EXCEEDED' 
                                    else ("memlim" if row[('state','b&p')] == 'MEM_EXCEEDED' 
                                          else "{:.1f}".format(row[('time','b&p')])), axis=1)
df3[('time (s)','gurobi')] = df2.apply(lambda row: "tilim" if row[('state','gurobi')] == 'TIME_EXCEEDED' 
                                       else ("memlim" if row[('state','gurobi')] == 'MEM_EXCEEDED' 
                                             else "{:.1f}".format(row[('time','gurobi')])), axis=1)

df3

instance  |V| density   n   m hval time (s)         nodes          lb  \
                                           b&p  gurobi   b&p gurobi  b&p   
0     david  803    0.61  78  87    3     12.5    37.2     3      1  2.0   
1      huck  599    0.46  72  74    6     16.7   622.1    11      1  2.0   
2      jean  495    0.38  66  80    4   1738.7   306.6  1365    645  3.0   
3   myciel3   40    0.39  11  11    2      0.0     0.0     1      1  2.0   
4   myciel4  142    0.40  23  23    2      0.1     1.2     1      1  2.0   
5   myciel5  472    0.39  47  47    2      3.1    61.0     1      1  2.0   
6  queen5_5  320    0.71  25  25    2      0.5    53.2     1      1  2.0   
7  queen6_6  580    0.66  36  36    3   2669.1  1161.5    41      1  2.0   
8  queen7_7  952    0.61  49  49    4    tilim  memlim    11      1  2.0   

          ub         
  gurobi b&p gurobi  
0    2.0   2      2  
1    2.0   2      2  
2    3.0   3      3  
3    2.0   2      2  
4    2.0   2      2  
5    2.0   2      2  
6    2.0   2      2  
7    2.0   2      2  
8    1.0   4      4

In [5]:
df3.to_latex("table11.tex", index=False, float_format="%.1f", na_rep="--")